# Marketing Analysis using XGBoost

This notebook demonstrates a complete marketing analytics workflow using a synthetic campaign dataset and an XGBoost classifier.


## 1. Import Libraries

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from data_preprocessing import generate_synthetic_marketing_data, build_preprocessor, split_features_target, RANDOM_STATE
from evaluation import plot_confusion_matrix, plot_roc_curve, plot_feature_importance


## 2. Generate and Inspect Dataset

The original project dataset is unavailable, so this notebook generates a realistic synthetic marketing campaign dataset with customer, channel, campaign, and engagement features.

In [ ]:
df = generate_synthetic_marketing_data(
    n_samples=800,
    output_path=PROJECT_ROOT / 'data' / 'raw' / 'synthetic_marketing_campaigns.csv'
)
df.head()

In [ ]:
df.info()

In [ ]:
df['conversion'].value_counts(normalize=True).rename('conversion_rate')

## 3. Train/Test Split and Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = split_features_target(df)
preprocessor = build_preprocessor(df)

X_train.shape, X_test.shape

## 4. XGBoost Model with GridSearchCV

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

param_grid = {
    'model__n_estimators': [30, 60],
    'model__max_depth': [2, 3],
    'model__learning_rate': [0.1],
    'model__subsample': [0.9],
    'model__colsample_bytree': [0.9],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train, y_train)
grid_search.best_params_, grid_search.best_score_

## 5. Model Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba),
}
pd.DataFrame([metrics]).round(4)

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

## 6. Visual Outputs

In [ ]:
plot_confusion_matrix(y_test, y_pred, PROJECT_ROOT / 'outputs' / 'confusion_matrix.png')
plot_roc_curve(y_test, y_proba, PROJECT_ROOT / 'outputs' / 'roc_curve.png')
plot_feature_importance(best_model, X_train, PROJECT_ROOT / 'outputs' / 'feature_importance.png')

print('Charts saved to outputs folder.')

## 7. Business Interpretation

The model can be used to identify customer and campaign characteristics that are most associated with conversion. In a real marketing team, this type of analysis could help prioritise high-performing channels, allocate budget more effectively, and create better customer targeting strategies.

Key practical applications include:

- Understanding which marketing channels drive conversion
- Segmenting customers based on likelihood to convert
- Improving retargeting campaigns
- Optimising discounts and campaign spend
- Supporting ROI-driven marketing decisions
